# M2 · Embedding ve Vektör Veritabanları
**Süre:** 20 dk canlı kodlama · **Demo çıktısı:** Dokümanları indeksleyen RAG hattı

| Dakika | Adım |
|---|---|
| 0-3 | Embedding nedir? Benzerlik ısı haritası |
| 3-8 | Dokümanları yükleme ve parçalama (chunking) |
| 8-14 | Embedding + Chroma'ya indeksleme |
| 14-18 | Retriever + LLM = RAG, kaynak gösteren cevap |
| 18-20 | Soru-cevap |

```
Dokümanlar → Parçala → Embedding → Vektör DB
                                         ↓  (benzerlik araması)
                   Soru → Embedding → En yakın k parça → LLM + bağlam → Kaynaklı cevap
```

In [ ]:
# ⚙️ Ortam hazırlığı: Colab'da ve yerel Jupyter'da çalışır
import os, sys, subprocess
REPO_URL = "https://github.com/volkanak/MBT-GenAI-Egitim.git"   # ← kendi GitHub reponuz
KOK = next((os.path.abspath(p) for p in [".", "..", "/content/MBT-GenAI-Egitim"]
            if os.path.exists(os.path.join(p, "ortak.py"))), None)
if KOK is None:  # Colab: repoyu klonla
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/MBT-GenAI-Egitim"], check=True)
    KOK = "/content/MBT-GenAI-Egitim"
try:
    import langgraph, chromadb, langchain_anthropic, sentence_transformers, langchain_mcp_adapters  # noqa
except ImportError:
    print("Paketler kuruluyor (1-3 dk)…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{KOK}/requirements.txt"], check=True)
sys.path.insert(0, KOK); os.chdir(KOK)
from ortak import *
api_anahtari_yukle()
print("Hazır ✔  Model:", MODEL)

## 1️⃣ Embedding: anlamı sayıya çevirmek
Aynı anlama gelen cümleler, **farklı kelimelerle** yazılsa da vektör uzayında birbirine yakındır.

In [ ]:
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import numpy as np

emb_model = SentenceTransformer(EMBEDDING_MODELI)

cumleler = [
    "Tedarikçiye avans ödemesi için teminat gerekir",
    "Satıcıya ön ödeme yaparken banka garantisi istenir",
    "Çalışanlar Salı ve Perşembe ofise gelir",
    "Kediler süt içmeyi sever",
]
V = emb_model.encode([f"query: {c}" for c in cumleler], normalize_embeddings=True)
print("Vektör boyutu:", V.shape)

benzerlik = V @ V.T  # normalize vektörlerde iç çarpım = kosinüs benzerliği
plt.imshow(benzerlik, cmap="Blues", vmin=0.6, vmax=1)
plt.xticks(range(4), [c[:18] + "…" for c in cumleler], rotation=30, ha="right")
plt.yticks(range(4), [c[:18] + "…" for c in cumleler])
for i in range(4):
    for j in range(4):
        plt.text(j, i, f"{benzerlik[i, j]:.2f}", ha="center", va="center")
plt.title("Kosinüs benzerliği"); plt.colorbar(); plt.show()

## 2️⃣ Dokümanları yükle ve parçala

In [ ]:
import pandas as pd

dokumanlar = dokumanlari_yukle()
pd.DataFrame(dokumanlar)[["id", "baslik", "departman", "konular"]]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ✍️ TODO: 400 karakterlik, 60 karakter örtüşmeli bir bölücü ile dokümanları parçalayın
...
print(len(dokumanlar), "doküman →", len(parcalar), "parça")
print(parcalar[0])

## 3️⃣ Embedding'le ve vektör veritabanına yaz (Chroma)  ✍️ *canlı kodlama*
> e5 modelleri dokümanlar için `passage:`, sorular için `query:` ön ekini bekler.

In [ ]:
import chromadb

# ✍️ TODO: Parçaları embedding'leyin ve kosinüs metrikli bir Chroma koleksiyonuna ekleyin
...
print("Koleksiyondaki kayıt:", koleksiyon.count())

## 4️⃣ Anlamsal arama (retriever)

In [ ]:
# ✍️ TODO: Soruyu embedding'leyip en yakın k parçayı döndüren ara() fonksiyonu
...

# "ön ödeme" kelimesi dokümanlarda geçmiyor, ama "avans" dokümanı bulunur 👇
for r in ara("Satıcıya ön ödeme yaparken nelere dikkat etmeliyim?"):
    print(f"{r['skor']:.3f}  {r['dok_id']}  {r['baslik']}")

## 5️⃣ RAG: bağlam + LLM = kaynaklı cevap  ✍️ *canlı kodlama*

In [ ]:
# ✍️ TODO: Bulunan parçalardan bağlam oluşturup modele kaynak göstermesini söyleyin
...

print(rag("Bir tedarikçiye 2.000.000 TL'lik sözleşme için %40 avans verebilir miyiz?"))

## 6️⃣ Halüsinasyon testi: dokümanlarda olmayan soru

In [ ]:
print(rag("Şirketin yemek kartı limiti ne kadar?"))

### ✅ Özet ve tartışma
- **Chunk boyutu** çok küçük → bağlam kopar; çok büyük → alakasız metin gelir. Deneyin: 200 / 800.
- Üretimde: kalıcı vektör DB (pgvector, Qdrant, Azure AI Search…), **hibrit arama** (BM25 + vektör), **re-ranking**.
- Bu kod `ortak.py` içinde `RagIndeksi` olarak duruyor → M3, M4, M5 tekrar kullanacak.
- **Eksik:** Sonuçlar herkese aynı sırayla geliyor. Kullanıcının rolü/ilgisi hesaba katılmıyor → **M3: GraphRAG**